In [ ]:
from google.colab import files
uploaded = files.upload()

#upload ur datset here

Saving synthetic_cybersecurity_dataset_v2.csv to synthetic_cybersecurity_dataset_v2.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, mean_squared_error, mean_absolute_error
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch

df = pd.read_csv("synthetic_cybersecurity_dataset_v2.csv")
feature_cols = ["Flow_Duration", "Total_Fwd_Packets", "Total_Backward_Packets",
                "Total_Length_of_Fwd_Packets", "Total_Length_of_Bwd_Packets",
                "Fwd_Packet_Length_Mean", "Bwd_Packet_Length_Mean",
                "Flow_Bytes_Per_Second", "Flow_Packets_Per_Second"]
X = df[feature_cols].values
y = df["Label"].values

timesteps = 10
def create_sequences(X, y, timesteps):
    X_seq, y_seq = [], []
    for i in range(len(X) - timesteps):
        X_seq.append(X[i:i + timesteps])
        y_seq.append(y[i + timesteps])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X, y, timesteps)
X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2,
                                                    random_state=42, shuffle=True, stratify=y_seq)

def seq_to_text(seq):
    text = "Over the past 10 seconds, "
    for i, timestep in enumerate(seq):
        text += f"at t-{9-i}: flow duration {int(timestep[0])} µs, " \
                f"fwd packets {int(timestep[1])}, bwd packets {int(timestep[2])}, " \
                f"flow bytes/s {int(timestep[7])}. "
    text += "Predict: normal (0) or anomaly (1)."
    return text

train_texts = [seq_to_text(seq) for seq in X_train]
test_texts = [seq_to_text(seq) for seq in X_test]

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=256)

class CyberDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = CyberDataset(train_encodings, y_train)
test_dataset = CyberDataset(test_encodings, y_test)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    predictions = (probs > 0.3).astype(int)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, pos_label=1, zero_division=0),
        "recall": recall_score(labels, predictions, pos_label=1, zero_division=0),
        "f1": f1_score(labels, predictions, pos_label=1, zero_division=0),
        "mse": mean_squared_error(labels, probs),
        "mae": mean_absolute_error(labels, probs),
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    evaluation_strategy="epoch",
    report_to="none"
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0]).to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

predictions = trainer.predict(test_dataset)
y_pred_prob = torch.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()
y_pred = (y_pred_prob > 0.3).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred_prob)
mae = mean_absolute_error(y_test, y_pred_prob)

print("\nEvaluation Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print("Prediction Probabilities (first 10):", y_pred_prob[:10])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mse,Mae
1,0.696700,0.643899,0.191919,0.191919,1.000000,0.322034,0.175042,0.397203
2,0.705200,0.645253,0.191919,0.191919,1.000000,0.322034,0.187311,0.420778
3,0.616500,0.643622,0.191919,0.191919,1.000000,0.322034,0.178673,0.404797
4,0.633400,0.644951,0.191919,0.191919,1.000000,0.322034,0.186396,0.419200
5,0.620300,0.645850,0.191919,0.191919,1.000000,0.322034,0.189071,0.423763



Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       160
           1       0.19      1.00      0.32        38

    accuracy                           0.19       198
   macro avg       0.10      0.50      0.16       198
weighted avg       0.04      0.19      0.06       198


Evaluation Metrics:
Accuracy: 0.1919
Precision: 0.1919
Recall: 1.0000
F1-Score: 0.3220
Mean Squared Error (MSE): 0.1891
Mean Absolute Error (MAE): 0.4238
Prediction Probabilities (first 10): [0.37622723 0.37630603 0.37625685 0.37620708 0.3761844  0.37616155
 0.37618804 0.37630066 0.37628594 0.37623245]


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
